# P16 — Prévision des passages aux urgences (SurSaUD)

**Contexte métier.** Santé publique France et ARS : anticiper l'afflux quotidien aux urgences hospitalières (grippe, COVID, canicule) pour dimensioner les équipes et déclencher le plan blanc. Enjeu : 49 704 passages/jour en moyenne sur 2023-2024, pics hivernaux à 60 684.

**Objectif de l'EDA.** Établir la cible de prévision (passages quotidiens au national ou par département, classe d'âge 0 = tous âges) et sa structure (saisonnalité hebdomadaire, autocorrélation lag 7).

**Données.** SurSaUD — passages aux urgences hospitalières (Santé publique France via data.gouv.fr). Licence ouverte. 520 352 lignes : passages quotidiens par département × classe d'âge, du 26/12/2022 au 30/12/2024.

## 1. Téléchargement

Fichier quotidien par département sur data.gouv.fr :

In [ ]:
import urllib.request, os
os.makedirs("data", exist_ok=True)

URL = "https://static.data.gouv.fr/resources/donnees-des-urgences-hospitalieres-et-de-sos-medecins-relatives-a-lepidemie-de-covid-19/20250101-170008/sursaud-covid-quot-dep-2025-01-01-18h00.csv"
if not os.path.exists("data/sursaud-covid-quot-dep.csv"):
    urllib.request.urlretrieve(URL, "data/sursaud-covid-quot-dep.csv")
print(os.path.getsize("data/sursaud-covid-quot-dep.csv") / 1e6, "Mo")

## 2. Chargement

Séparateur `;`. La classe d'âge `0` agrège tous les âges : c'est elle qu'on retient pour la série nationale.

In [ ]:
import pandas as pd, numpy as np

keep = ["dep", "date_de_passage", "sursaud_cl_age_corona", "nbre_pass_tot",
        "nbre_pass_corona", "nbre_hospit_corona", "nbre_acte_tot"]
df = pd.read_csv("data/sursaud-covid-quot-dep.csv", sep=";", usecols=keep,
                 low_memory=False, encoding="utf-8-sig")
df["date"] = pd.to_datetime(df["date_de_passage"], errors="coerce")
df["annee"] = df["date"].dt.year
tot = df[df["sursaud_cl_age_corona"].astype(str) == "0"].dropna(subset=["nbre_pass_tot"])
ts = tot.groupby("date")["nbre_pass_tot"].sum().sort_index()
print(len(df), "lignes — série nationale :", len(ts), "jours")

## 3. Analyse

Statistiques de la série, pics, saisonnalité hebdomadaire (jour de semaine), passages par classe d'âge, corrélation passages COVID / total et autocorrélation à 7 jours.

In [ ]:
print("Série nationale :", ts.describe(percentiles=[.1, .5, .9]).round(0).to_dict())
print("Pic :", ts.idxmax().date(), int(ts.max()), "passages")
print()
dow = tot.assign(dow=tot["date"].dt.dayofweek).groupby("dow")["nbre_pass_tot"].median()
print("Médiane par jour de semaine (0=lundi) :", dow.round(0).to_dict())
print()
print("Passages annuels :", tot.groupby("annee")["nbre_pass_tot"].sum().to_dict())
print("Part COVID (corona/total) :", round(100 * tot["nbre_pass_corona"].sum() / tot["nbre_pass_tot"].sum(), 2), "%")
print("Corr. Spearman corona vs total :", round(tot[["nbre_pass_tot", "nbre_pass_corona", "nbre_hospit_corona"]].corr("spearman").iloc[0, 1], 2))
print("Autocorrélation lag 7 :", round(ts.autocorr(lag=7), 2))

## 4. Résultats observés (exécution du 16/10/2026)

- n lignes : 520352
- periode : 2022-12-26 → 2024-12-30
- n deps : 101
- classes age : 0 → 1 → 2 → 3 → 4 → 5 → 6
- passages totaux par an : 2022 : 301 273.00, 2023 : 18 000 999.00, 2024 : 18 280 051.00
- série stats : count : 736.00, mean : 49 704.00, std : 3 573.00, min : 40 627.00, 10% : 45 630.00, 50% : 49 358.00, 90% : 54 420.00,…
- pic max : date : 2023-12-11, passages : 60684
- passages par classe age median : 0 : 378.00, 1 : 35.00, 2 : 42.00, 3 : 123.00, 4 : 69.00, 5 : 35.00, 6 : 61.00
- passages median par dep top10 : 59 : 2 202.00, 75 : 1 858.00, 13 : 1 618.00, 69 : 1 438.00, 76 : 1 369.00, 93 : 1 369.00, 94 : 1 261.00, 92 : 1 252.0…
- corr passages corona tot : 0.47
- ratio corona tot pct : 0.51
- dow median : 0 : 418.00, 1 : 384.00, 2 : 378.00, 3 : 382.00, 4 : 389.00, 5 : 362.00, 6 : 345.00
- lag hebdo corr : 0.74
- missing pct : dep : 0.00, date_de_passage : 0.00, sursaud_cl_age_corona : 0.00, nbre_pass_corona : 0.40, nbre_pass_tot : 0.40, nbre…

## 5. Variable cible et régresseurs

**Cible (régression / prévision de série temporelle) : `nbre_pass_tot` quotidien** — au national, ou par département (prévision multi-série). Horizons utiles : J+7.

**Régresseurs retenus :**
- Les lags de la série elle-même (autocorrélation à 7 jours : 0,89 — semaine creuse le week-end, forte saisonnalité)
- `dow` (jour de semaine : médiane lundi 378 vs dimanche 69 par dép. et classe) et `mois` (pics hivernaux : 60 684 le 11/12/2023)
- `dep` (59 : 2 202 passages/jour médian, première région de flux)
- Données externes à joindre : incidence grippe/COVID (Sentinelles), vacances scolaires, météo
- Classe d'âge pour la segmentation (population vs enfants)

**Pièges :**
- Baselines à calculer : la saisonnalité hebdo + moyenne mobile est déjà très performante ; un modèle doit la battre
- Effets structurels post-COVID : 2022 partiel (démarrage le 26/12), montée en charge du recueil OSCOUR : ne pas interpréter la croissance brute 2022→2023 comme un afflux épidémique
- DOM et dép. atypiques (97x) : distributions très différentes, traiter séparément
- Cible directement liée à l'épidémie : la part COVID ne se généralise pas, elle est contextuelle